In [ ]:
import os
import xml.etree.ElementTree as ET
import pandas as pd

In [ ]:
root = "OhioDataSet"
years = ["2018", "2020"]
subfolder = "train"

output_folder = "DatasetsCSV"

In [ ]:
def parse_section(root, tag):
    """Parses an XML section (e.g. glucose_level, basal) into a DataFrame."""
    section = root.find(tag)
    if section is None:
        return pd.DataFrame()

    rows = [elem.attrib for elem in section]
    return pd.DataFrame(rows)

In [ ]:
def load_patient_xml(path_xml):
    """Loads a patient XML file and returns a dict of DataFrames, one per signal."""
    tree = ET.parse(path_xml)
    root = tree.getroot()

    return {
        "glucose": parse_section(root, "glucose_level"),
        "basal": parse_section(root, "basal"),
        "bolus": parse_section(root, "bolus"),
        "meal": parse_section(root, "meal"),
        "exercise": parse_section(root, "exercise"),
    }

In [ ]:
conversion_rules = {
    "glucose": {
        "columns": {
            "ts": lambda s: pd.to_datetime(s, format="%d-%m-%Y %H:%M:%S", errors="coerce"),
            "value": lambda s: pd.to_numeric(s, errors="coerce").astype("Int64"),
        }
    },
    "basal": {
        "columns": {
            "ts": lambda s: pd.to_datetime(s, format="%d-%m-%Y %H:%M:%S", errors="coerce"),
            "value": lambda s: pd.to_numeric(s, errors="coerce").astype(float),
        }
    },
    "bolus": {
        "columns": {
            "ts": lambda s: pd.to_datetime(s, format="%d-%m-%Y %H:%M:%S", errors="coerce"),
            "dose": lambda s: pd.to_numeric(s, errors="coerce").astype(float),
            "bwz_carb_input": lambda s: pd.to_numeric(s, errors="coerce").astype("Int64"),
        }
    },
    "meal": {
        "columns": {
            "ts": lambda s: pd.to_datetime(s, format="%d-%m-%Y %H:%M:%S", errors="coerce"),
            "carbs": lambda s: pd.to_numeric(s, errors="coerce").astype("Int64"),
        }
    },
    "exercise": {
        "columns": {
            "ts": lambda s: pd.to_datetime(s, format="%d-%m-%Y %H:%M:%S", errors="coerce"),
            "intensity": lambda s: pd.to_numeric(s, errors="coerce").astype("Int64"),
            "duration": lambda s: pd.to_numeric(s, errors="coerce").astype("Int64"),
            "competitive": lambda s: s.astype(str),
        }
    },
}

In [ ]:
def apply_transformations(signal_type, df):
    if df.empty:
        return df.copy()

    rules = conversion_rules.get(signal_type, {}).get("columns", {})
    df = df.copy()

    # Bolus: rename ts_begin to ts and drop ts_end
    if signal_type == "bolus":
        if "ts_end" in df.columns:
            df = df.drop(columns=["ts_end"])
        if "ts_begin" in df.columns:
            df.rename(columns={"ts_begin": "ts"}, inplace=True)

    if signal_type == "exercise":
        df = df.drop(columns=["type", "competitive"], errors="ignore")

        # Combine duration and intensity into a single exercise load value
        if "duration" in df.columns and "intensity" in df.columns:
            df["value"] = pd.to_numeric(df["duration"], errors="coerce") * \
                        pd.to_numeric(df["intensity"], errors="coerce")

        df = df.drop(columns=["duration", "intensity"], errors="ignore")

    for col, transform in rules.items():
        if col in df.columns:
            df[col] = transform(df[col])

    return df

In [ ]:
def save_csv(df, patient, signal_type):
    if df.empty:
        return

    out_dir = os.path.join(output_folder, subfolder)
    os.makedirs(out_dir, exist_ok=True)

    out_path = os.path.join(out_dir, f"{signal_type}_{patient}.csv")
    df.to_csv(out_path, index=False, encoding="utf-8")

In [ ]:
def process_dataset():
    for year in years:
        base_path = os.path.join(root, year, subfolder)

        if not os.path.isdir(base_path):
            continue

        for filename in os.listdir(base_path):
            if filename.endswith("-ws-training.xml"):
                patient = filename.split("-")[0]
                full_path = os.path.join(base_path, filename)

                data = load_patient_xml(full_path)

                for signal_type, df in data.items():
                    df_transformed = apply_transformations(signal_type, df)
                    save_csv(df_transformed, patient, signal_type)

                print(f"Patient {patient} processed.")

    print("Complete.")

In [ ]:
process_dataset()